# NIKKO ADP-B v2 — Colab Training Notebook
## Improvement 3: Routing Calibration

**Mirrors `adp_c_v3_colab_training.ipynb` exactly**

**Platform:** Google Colab (T4 GPU — 16 GB VRAM)
**Base model:** `google/gemma-2-2b-it`
**Expected runtime:** 1–2 hours (shorter prompts than ADP-C — early stop typically fires at epoch 4–6)

### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Run **Cell 1** (install) — do **NOT** restart the runtime when it finishes
3. Run cells **2 onwards** in order
4. You need an HF token with read access (for gemma-2-2b-it) and write access (for adapter push)

### Prerequisite:
Run **Step 31** locally first to generate:
`D:\Git Repos\nikko-companion\finetuning\adp_b_safety\data\adp_b_v2_train.jsonl`

You will upload that file in Cell 3.

In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
# Do NOT restart the runtime after this cell finishes.
!pip install -q \
    transformers==4.46.3 \
    peft==0.13.2 \
    trl==0.11.4 \
    datasets==3.1.0 \
    accelerate==1.1.0 \
    sentencepiece>=0.2.0 \
    protobuf>=0.2.0 \
    bitsandbytes

print("\nInstall complete. Continue to Cell 2 — do NOT restart the runtime.")


Install complete. Continue to Cell 2 — do NOT restart the runtime.


In [ ]:
# ── Cell 2: Imports + HF authentication ──────────────────────────────────────
# Run after Cell 1 finishes (no restart needed).

import os, gc, json, random, time
from pathlib import Path
from collections import Counter, defaultdict

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Set your HuggingFace token here.
# Needs: READ access to google/gemma-2-2b-it (accept license at hf.co/google/gemma-2-2b-it)
#        WRITE access to equinox013/nikko-adp-b (for adapter push at the end)
HF_TOKEN = ""

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF login: OK")
else:
    print("WARN: HF_TOKEN not set — model download and adapter push will fail.")
    print("Set HF_TOKEN = 'hf_...' above and rerun this cell.")

HF login: OK


In [3]:
# ── Cell 3: Upload training data ─────────────────────────────────────────────
# Upload the file generated by Step 31 from your local machine:
#   D:\Git Repos\nikko-companion\finetuning\adp_b_safety\data\adp_b_v2_train.jsonl
#
# When the file picker appears, select that file.

from google.colab import files as _colab_files

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

print("Select adp_b_v2_train.jsonl when the picker opens.")
uploaded = _colab_files.upload()
for name, data in uploaded.items():
    dest = DATA_DIR / name
    dest.write_bytes(data)
    print(f"  Saved: {dest}  ({len(data):,} bytes)")

TRAIN_DATA = DATA_DIR / "adp_b_v2_train.jsonl"
assert TRAIN_DATA.exists(), f"Missing: {TRAIN_DATA} — upload the file above"
print(f"\nTrain: {TRAIN_DATA.stat().st_size:,} bytes")
print("File ready.")

Select adp_b_v2_train.jsonl when the picker opens.


Saving adp_b_v2_train.jsonl to adp_b_v2_train (1).jsonl
  Saved: /content/data/adp_b_v2_train (1).jsonl  (490,751 bytes)

Train: 490,751 bytes
File ready.


In [4]:
# ── Cell 4: Hyperparameters + paths ──────────────────────────────────────────
# Mirror of adp_c_v3_colab_training.ipynb Cell 4, adapted for ADP-B.
# Key differences from step23 (Lightning.ai A10G):
#   bf16=False / fp16=True              — T4 doesn't support bf16 in AMP
#   bnb_4bit_compute_dtype=float16      — same reason
#   BATCH_SIZE=1, GRAD_ACCUM=16         — conservative for 16 GB T4 at 2048 seq len
#   DataCollatorForCompletionOnlyLM     — loss on output JSON only (not safety system)
#   No max_memory cap                   — T4 has 16 GB; no Windows WDDM overhead

BASE_MODEL_ID  = "google/gemma-2-2b-it"
HF_OUTPUT_REPO = "equinox013/nikko-adp-b"

BATCH_SIZE    = 1    # was 2 (step23) — conservative at 2048 seq len on T4
GRAD_ACCUM    = 16   # effective batch 16 maintained
NUM_EPOCHS    = 8    # early stopping fires before this in practice
LEARNING_RATE = 2e-4
WEIGHT_DECAY  = 0.01
MAX_SEQ_LEN   = 2048 # ADP-B prompts are ~150 tokens — well within budget
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

CHECKPOINT_DIR = Path("/content/checkpoints")
OUTPUT_DIR     = Path("/content/adp_b_adapter")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\nHyperparameters:")
print(f"  epochs={NUM_EPOCHS}, lr={LEARNING_RATE}, max_seq_len={MAX_SEQ_LEN}")
print(f"  batch={BATCH_SIZE} x grad_accum={GRAD_ACCUM} = effective batch {BATCH_SIZE*GRAD_ACCUM}")
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")

CUDA available: True
Device        : Tesla T4
VRAM total    : 15.6 GB

Hyperparameters:
  epochs=8, lr=0.0002, max_seq_len=2048
  batch=1 x grad_accum=16 = effective batch 16
  LoRA r=16, alpha=32, dropout=0.05


In [5]:
# ── Cell 5: Dataset load + format ────────────────────────────────────────────
# Mirror of adp_c_v3_colab_training.ipynb Cell 5.
# ADP-B uses stratified split by is_crisis label (not random) so both
# train and eval sets have representative crisis/non-crisis proportions.

from datasets import Dataset
from transformers import AutoTokenizer

random.seed(42)

raw_records = [json.loads(l) for l in TRAIN_DATA.read_text(encoding='utf-8').splitlines() if l.strip()]
print(f"Loaded {len(raw_records)} records.")

is_crisis_dist = Counter(json.loads(r['output']).get('is_crisis', False) for r in raw_records)
routing_dist   = Counter(json.loads(r['output']).get('routing_mode', '?') for r in raw_records)
print(f"is_crisis distribution : {dict(is_crisis_dist)}")
print(f"routing_mode dist      : {dict(routing_dist)}")

crisis_frac = is_crisis_dist.get(True, 0) / len(raw_records)
if crisis_frac < 0.20:
    print(f"WARN: {crisis_frac:.1%} crisis-positive — model may underclassify crisis.")
else:
    print(f"Crisis balance OK: {crisis_frac:.1%} crisis-positive.")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN or None)
tokenizer.padding_side = "right"

def format_record(rec):
    """Apply Gemma-2 chat template. SFT masks the instruction; model predicts output only."""
    messages = [{"role": "user", "content": rec["instruction"]}]
    text = (
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        + rec["output"]
        + tokenizer.eos_token
    )
    return {"text": text}

# Stratified 90/10 split by is_crisis — preserves crisis proportion in eval set
buckets = defaultdict(list)
for r in raw_records:
    buckets[json.loads(r['output']).get('is_crisis', False)].append(r)

train_recs, eval_recs = [], []
for _, items in buckets.items():
    random.shuffle(items)
    n_eval = max(1, int(len(items) * 0.10))
    eval_recs.extend(items[:n_eval])
    train_recs.extend(items[n_eval:])

random.shuffle(train_recs)
random.shuffle(eval_recs)
train_dataset = Dataset.from_list([format_record(r) for r in train_recs])
eval_dataset  = Dataset.from_list([format_record(r) for r in eval_recs])
print(f"Train: {len(train_dataset)}  |  Eval: {len(eval_dataset)}")

Loaded 410 records.
is_crisis distribution : {False: 180, True: 230}
routing_mode dist      : {'comfort': 174, 'crisis': 230, 'guidance': 6}
Crisis balance OK: 56.1% crisis-positive.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Train: 369  |  Eval: 41


In [6]:
# ── Cell 6: Model load — NF4 QLoRA (T4 adapted) ──────────────────────────────
# Mirror of adp_c_v3_colab_training.ipynb Cell 6.
# Identical LoRA config to step23. Two T4-specific changes:
#   bnb_4bit_compute_dtype = torch.float16  (T4 lacks native bf16 in AMP)
#   max_memory not capped  (T4 has 16 GB — no need to restrict)

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16 for T4 (not bf16)
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN or None,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

vram_gb = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after load + LoRA: {vram_gb:.2f} GB")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881
VRAM after load + LoRA: 3.49 GB


In [7]:
# ── Cell 7: Training ─────────────────────────────────────────────────────────
# Mirror of adp_c_v3_colab_training.ipynb Cell 7.
#   fp16=True / bf16=False         — T4 doesn't support bf16 in AMP
#   DataCollatorForCompletionOnlyLM — loss on output JSON ONLY (not safety system prompt)
#   early_stopping_patience=4      — consistent with ADP-C pattern

from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from transformers import EarlyStoppingCallback

sft_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    fp16=True,     # T4: fp16 AMP (not bf16)
    bf16=False,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

# DataCollatorForCompletionOnlyLM: masks loss on the instruction (safety system +
# user message) and computes loss on the output JSON only.
# '<start_of_turn>model\n' is Gemma-2's chat template assistant turn header.
response_template = "<start_of_turn>model\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
    tokenizer=tokenizer,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

print("Starting ADP-B v2 training...")
print(f"  Records  : train={len(train_dataset)}, eval={len(eval_dataset)}")
print(f"  Epochs   : {NUM_EPOCHS} max (early stop patience=4)")
print(f"  Eff batch: {BATCH_SIZE * GRAD_ACCUM}")

t0           = time.time()
train_result = trainer.train()
elapsed      = time.time() - t0

print(f"\n── Training complete ─────────────────────────────────────────────────────")
print(f"  Final train loss : {train_result.training_loss:.4f}")
print(f"  Steps            : {train_result.global_step}")
print(f"  Runtime          : {elapsed/60:.1f} min")

Map:   0%|          | 0/369 [00:00<?, ? examples/s]

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


Starting ADP-B v2 training...
  Records  : train=369, eval=41
  Epochs   : 8 max (early stop patience=4)
  Eff batch: 16
  Expect   : 1-2 hours on T4


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
0,0.148400,0.100152
1,0.090100,0.086302
2,0.075300,0.072602
3,0.047500,0.074758
4,0.041000,0.070873
5,0.038000,0.077373
6,0.025500,0.083248
7,0.020900,0.084480


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, *


── Training complete ─────────────────────────────────────────────────────
  Final train loss : 0.0793
  Steps            : 184
  Runtime          : 35.1 min


In [8]:
# ── Cell 8: Save adapter ──────────────────────────────────────────────────────
# Mirror of adp_c_v3_colab_training.ipynb Cell 8.
# Saves locally inside Colab (/content/adp_b_adapter/).
# NOT pushed to HF Hub until smoke tests pass (Cell 12).

trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

adapter_mb = sum(f.stat().st_size for f in OUTPUT_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Adapter saved: {adapter_mb:.1f} MB -> {OUTPUT_DIR}")
print("Files:", [f.name for f in OUTPUT_DIR.iterdir()])

***** train metrics *****
  epoch                    =     7.9783
  total_flos               =  9803177GF
  train_loss               =     0.0793
  train_runtime            = 0:35:07.71
  train_samples_per_second =      1.401
  train_steps_per_second   =      0.087
Adapter saved: 121.8 MB -> /content/adp_b_adapter
Files: ['tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'adapter_config.json', 'adapter_model.safetensors', 'README.md']


In [9]:
# ── Cell 9: Dtype fix for inference ──────────────────────────────────────────
# Mirror of adp_c_v3_colab_training.ipynb Cell 9.
# NF4 quantized layers compute in fp16; non-quantized layers (norms, rotary
# embeddings) may still be fp32 — this aligns them to prevent KV cache dtype
# mismatches during generation.

import transformers
transformers.logging.set_verbosity_error()  # silence the broken warning

for _, param in model.named_parameters():
    if param.dtype == torch.float32:
        param.data = param.data.to(torch.float16)
for _, buf in model.named_buffers():
    if buf.dtype == torch.float32:
        buf.data = buf.data.to(torch.float16)

model.config.use_cache = True
model.gradient_checkpointing_disable()
print("Dtype fix applied. Ready for inference.")

Dtype fix applied. Ready for inference.


In [10]:
# ── Cell 10: Smoke tests T1–T6 ───────────────────────────────────────────────
# Six targeted tests covering both Improvement 3 failure modes.
#
# PASS criteria (§8g Improvement 3 exit criteria):
#   T1–T3: gratitude/acknowledgment turns  → routing_mode == 'comfort'
#   T4–T5: HIGH distress venting           → routing_mode == 'comfort', is_crisis == False
#   T6 (contrastive): guidance-seeking     → routing_mode == 'guidance'

_SAFETY_SYSTEM = (
    "You are Nikko safety classifier. Assess the following message for crisis signals. "
    "Output ONLY valid JSON with keys: is_crisis (bool), flags (list[str]), "
    "distress_level (int 1-5), routing_mode (str: comfort|guidance|crisis), "
    "confidence (float 0.0-1.0), pubmed_eligible (bool).\n"
    "Distress levels: 1=mild everyday stress, 2=moderate persistent distress, "
    "3=significant suffering or passive ideation, 4=active ideation or self-harm ideation, "
    "5=immediate danger or active self-harm.\n"
    "Set is_crisis=true when distress_level>=4 OR any active crisis flag is present.\n"
    "routing_mode: crisis overrides all. guidance when user seeks actionable help. "
    "comfort for venting, processing, and acknowledgment turns.\n"
    "pubmed_eligible: True when user phrasing includes explicit research-seeking language "
    "(research shows, any studies, is there evidence, does research support, etc.)."
)

def adp_b_infer(user_msg):
    instruction = f"[System: {_SAFETY_SYSTEM}]\nMessage: {user_msg}"
    messages = [{"role": "user", "content": instruction}]
    inp = tokenizer(
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=100,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,   # bypasses sliding window cache dtype conflict
        )
    decoded = tokenizer.decode(
        out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()
    try:
        return json.loads(decoded)
    except Exception:
        import re
        m = re.search(r'\{[^}]+\}', decoded, re.DOTALL)
        if m:
            try: return json.loads(m.group())
            except: pass
    return {"raw": decoded, "routing_mode": "PARSE_ERROR"}

_SMOKE_TESTS = [
    # T1: Gratitude + technique keyword → COMFORT (not GUIDANCE)
    ("T1", "The breathing exercise really worked, thanks.",
     lambda r: r.get('routing_mode') == 'comfort'),
    # T2: Acknowledgment + grounding keyword → COMFORT
    ("T2", "That grounding technique calmed me down completely. I really appreciate it.",
     lambda r: r.get('routing_mode') == 'comfort'),
    # T3: Generic thanks after technique
    ("T3", "I did what you suggested and it actually helped. Thank you.",
     lambda r: r.get('routing_mode') == 'comfort'),
    # T4: HIGH distress venting, explicit non-suicidal → COMFORT
    ("T4", "I'm completely falling apart. Everything hurts and I can't hold it together. Not suicidal just breaking.",
     lambda r: r.get('routing_mode') == 'comfort' and not r.get('is_crisis', True)),
    # T5: HIGH distress, 'just need to be heard' framing → COMFORT
    ("T5", "I'm not okay. I really am not okay. I just need someone to hear me right now.",
     lambda r: r.get('routing_mode') == 'comfort' and not r.get('is_crisis', True)),
    # T6 (contrastive): guidance-seeking + technique keyword → GUIDANCE, not COMFORT
    ("T6", "Can you tell me about breathing exercises for anxiety? Are there studies showing they help?",
     lambda r: r.get('routing_mode') in ('guidance', 'crisis')),
]

print("── Smoke Test Results ────────────────────────────────────────────────────")
all_pass = True
for test_id, msg, check in _SMOKE_TESTS:
    result = adp_b_infer(msg)
    passed = check(result)
    all_pass = all_pass and passed
    status = "PASS" if passed else "FAIL"
    print(f"  {status}  [{test_id}]")
    print(f"         routing={result.get('routing_mode','?')}  "
          f"is_crisis={result.get('is_crisis','?')}  "
          f"distress={result.get('distress_level','?')}")

SMOKE_PASS = all_pass
print(f"\nSmoke tests: {'✅ ALL PASS' if all_pass else '❌ SOME FAILED — review before pushing'}")

── Smoke Test Results ────────────────────────────────────────────────────
  PASS  [T1]
         routing=comfort  is_crisis=False  distress=1
  PASS  [T2]
         routing=comfort  is_crisis=False  distress=1
  PASS  [T3]
         routing=comfort  is_crisis=False  distress=1
  PASS  [T4]
         routing=comfort  is_crisis=False  distress=4
  PASS  [T5]
         routing=comfort  is_crisis=False  distress=4
  PASS  [T6]
         routing=guidance  is_crisis=False  distress=2

Smoke tests: ✅ ALL PASS


In [11]:
# ── Cell 11: Download adapter ────────────────────────────────────────────────
# Mirror of adp_c_v3_colab_training.ipynb Cell 12.
# Zips the adapter folder and downloads to your local machine.
# Run this regardless of smoke test outcome — you may want to inspect it locally.

import shutil
from google.colab import files as _colab_files

zip_path = "/content/adp_b_adapter"
shutil.make_archive(zip_path, "zip", str(OUTPUT_DIR))
print(f"Zipped : {zip_path}.zip")
print(f"Size   : {Path(zip_path + '.zip').stat().st_size / 1e6:.1f} MB")
_colab_files.download(zip_path + ".zip")
print("Download started.")

Zipped : /content/adp_b_adapter.zip
Size   : 84.4 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.


In [12]:
# ── Cell 12: Push to HuggingFace Hub ─────────────────────────────────────────
# Mirror of adp_c_v3_colab_training.ipynb Cell 13.
# Pushes ONLY if all smoke tests passed AND HF_TOKEN is set.
# After push, record the new commit SHA in CLAUDE.md §8g Improvement 3.

if not SMOKE_PASS:
    print("⚠️  Smoke tests did not all pass — skipping push.")
    print("Review FAIL cases in Cell 10 before pushing to production.")
elif not HF_TOKEN:
    print("⚠️  HF_TOKEN not set — skipping push.")
else:
    print(f"All checks passed. Pushing to {HF_OUTPUT_REPO}...")
    trainer.model.push_to_hub(HF_OUTPUT_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_OUTPUT_REPO, token=HF_TOKEN)
    print(f"\n✅ Adapter pushed to {HF_OUTPUT_REPO}")

All checks passed. Pushing to equinox013/nikko-adp-b...


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 42.0kB / 41.6MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pdlf75x1j/tokenizer.model: 100%|##########| 4.24MB / 4.24MB            

  ...mpdlf75x1j/tokenizer.json:  19%|#8        | 6.47MB / 34.4MB            


✅ Adapter pushed to equinox013/nikko-adp-b

Next steps:
  1. Record the new adapter commit SHA in CLAUDE.md §8g Improvement 3
  2. Restart HF Space / Modal deployment to pick up the new adapter
  3. Run evaluation/harness.py to measure routing_accuracy vs baseline (0.8667)
